In [ ]:
# I hand typed this all after watching the video, no claude at all. Come at me.

# ...but, I mean, I had to google a few things. My math chops are too rusty to
# do the derivative of tanh on the spot. Whatever, I'm still smart.

# Also the video used tanh everywhere, as far as I could tell, which means its
# incapable of outputs > 1 or < -1, and that's easy to fix, so I changed that.
# Besides that and some mostly arbitrary reorganizations of things, this should
# be pretty darn similar to the video.

# Also I hate notebooks, but I've decided to torture myself by getting used to
# them. So this is a single notebook instead of multiple files.

In [3]:
import math
import numpy as np
import random
from graphviz import Digraph
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
class Gradable:
  def parameters(self):
    return []

  def zero_grad(self):
    for p in self.parameters():
      p.grad = 0

In [5]:
class Value:
  def __init__(self, data, children=(), label="", op=""):
    self.data = data
    self._children = set(children)
    self._label = label
    self.grad = 0
    self._backward = lambda: None

    # only for graphviz/debugging, backward pass info captured at creation
    # via setting _backward in the operation that created this Value
    self._op = op


  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, children=(self, other), op="+")

    def _backward():
      self.grad += out.grad
      other.grad += out.grad
    
    out._backward = _backward
    return out
        
  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, children=(self, other), op="*")
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    
    out._backward = _backward
    return out

  def __pow__(self, other):
    assert isinstance(other, (int, float)), "only floats and ints are supported"
    out = Value(self.data ** other, children=(self,), op=f"**{other}")

    def _backward():
      self.grad += other * self.data**(other-1) * out.grad

    out._backward = _backward
    return out

  def _topo(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._children:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    return topo

  def backward_from(self):
    nodes = self._topo()
    for n in reversed(nodes):
      n._backward()
    
  def _trace(self):
    nodes, edges = set(), set()
    def build(v):
      if v not in nodes:
        nodes.add(v)
        for child in v._children:
          edges.add((child, v))
          build(child)
    build(self)
    return nodes, edges
    
  def draw_from(self):
    dot = Digraph(format='svg', graph_attr={'rankdir':'LR'})

    nodes, edges = self._trace()
    for n in nodes:
      uid = str(id(n))
      dot.node(name = uid, label = f"{{{n._label} | {n.data:.4f} | grad: {n.grad:.2e}}}", shape='record')
      if n._op:
        dot.node(name = uid + n._op, label = n._op)
        dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
      dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot
  
  def __neg__(self):
    return self * -1

  def __radd__(self, other):
    return self + other

  def __sub__(self, other):
    return self + (-other)

  def __rsub__(self, other):
    return other + (-self)
    
  def __rmul__(self, other):
    return self * other
  
  def __truediv__(self, other):
    return self * other**-1

  def __rtruediv__(self, other):
    return other * self**-1
    
  def __repr__(self):
    return f"Value(data={self.data}, grad={self.grad})"

### Activation Functions

def tanh(v):
  x = v.data
  t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
  out = Value(t, children=(v,), op="tanh")

  def _backward():
    v.grad += (1 - t**2) * out.grad
  
  out._backward = _backward
  return out

def relu(v):
  out = Value(max(0, v.data), children=(v,), op=f"relu")

  def _backward():
    v.grad += (1 if v.data > 0 else 0) * out.grad
  
  out._backward = _backward
  return out

def ident(v):
  return v

In [6]:
class Neuron(Gradable):
  def __init__(self, ins, act_func):
    self.ws = [Value(random.uniform(-0.1, 0.1)) for _ in range(ins)]
    self.b = Value(random.uniform(-0.1,0.1))
    self.act_func = act_func

  def __call__(self, xs):
    assert len(xs) == len(self.ws)
    return self.act_func(sum([wn*xn for wn, xn in zip(self.ws, xs)], self.b))

  def parameters(self):
    return self.ws + [self.b]

  def __repr__(self):
    return f"Neuron({len(self.ws)})({self.act_func.__name__})"

In [7]:
class Layer(Gradable):
  def __init__(self, nin, nout, act_func=relu):
    self.neurons = [Neuron(nin, act_func) for _ in range(nout)]

  def __call__(self, xs):
    return [n(xs) for n in self.neurons]

  def parameters(self):
    return [p for n in self.neurons for p in n.parameters()]

  def __repr__(self):
    return f"Layer[{ ','.join(str(n) for n in self.neurons) }]"

In [8]:
class MLP(Gradable):
  def __init__(self, nin, layer_sizes):
    assert len(layer_sizes) > 1
    # First layer from inputs
    self.layers = [Layer(nin, layer_sizes[0])]

    # Internal layers, all relu
    self.layers.extend([Layer(layer_sizes[i-1], layer_sizes[i]) for i in range(1,len(layer_sizes)-1)])

    # Last layer, no activation function
    self.layers.append(Layer(layer_sizes[-2], layer_sizes[-1], tanh))

  def __call__(self, xs):
      for layer in self.layers:
        xs = layer(xs)
      return xs

  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]
    
  def __repr__(self):
    return f"MLP[{'; '.join(str(layer) for layer in self.layers)}]"

In [9]:
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]
nn = MLP(3, [4, 4, 1])

def loss_func(out, y):
  return (out - y)**2

training_speed = 0.1

results = [nn(x)[0] for x in xs]
loss = sum([loss_func(result, y) for result, y in zip(results, ys)])
print(f"loss:{loss.data:.4f}, results:{[f'{result.data:.4f}' for result in results]}")

for _ in range(100):
  loss.grad = 1
  loss.backward_from()
  
  for p in nn.parameters():
    p.data += -training_speed * p.grad

  nn.zero_grad()
  results = [nn(x)[0] for x in xs]
  loss = sum([loss_func(result, y) for result, y in zip(results, ys)])
  print(f"loss:{loss.data:.4f}, results:{[f'{result.data:.4f}' for result in results]}")

loss:4.0060, results:['-0.0017', '0.0006', '0.0010', '0.0003']
loss:4.0035, results:['-0.0017', '-0.0004', '0.0004', '-0.0001']
loss:4.0015, results:['-0.0013', '-0.0008', '0.0004', '0.0001']
loss:3.9999, results:['-0.0008', '-0.0010', '0.0005', '0.0004']
loss:3.9984, results:['-0.0004', '-0.0012', '0.0007', '0.0007']
loss:3.9969, results:['-0.0000', '-0.0015', '0.0008', '0.0010']
loss:3.9953, results:['0.0003', '-0.0018', '0.0010', '0.0013']
loss:3.9932, results:['0.0007', '-0.0022', '0.0011', '0.0016']
loss:3.9906, results:['0.0012', '-0.0028', '0.0013', '0.0020']
loss:3.9872, results:['0.0017', '-0.0036', '0.0015', '0.0026']
loss:3.9826, results:['0.0024', '-0.0048', '0.0017', '0.0032']
loss:3.9753, results:['0.0032', '-0.0070', '0.0018', '0.0040']
loss:3.9653, results:['0.0079', '-0.0065', '0.0057', '0.0087']
loss:3.9499, results:['0.0070', '-0.0139', '0.0033', '0.0076']
loss:3.9318, results:['0.0144', '-0.0137', '0.0093', '0.0156']
loss:3.9001, results:['0.0116', '-0.0291', '0.001